# Tabular regression with `LanguageModelRegressor`

The shared fine-tune (see `00-intro.ipynb`) lets the model condition any column on the
others. For regression the target is a number, and the model defines a distribution $p(y \mid x)$
over the *text* of that number.

## Why we sample and average

Greedy decoding returns the most likely token sequence — the **mode** of $p(y \mid x)$ — which is
not what a regressor wants. The predictor that minimizes squared error is the **conditional
mean** $\mathbb{E}[y \mid x]$, and mode $\neq$ mean for any skewed conditional.

So `LanguageModelRegressor.predict` draws $n$ independent completions
$y^{(1)},\dots,y^{(n)} \sim p(y \mid x)$ (one per sampled column order) and returns their average —
a **Monte-Carlo estimate** of the conditional mean:

$$ \hat{y} = \frac{1}{n}\sum_{k=1}^{n} y^{(k)} \;\xrightarrow[n\to\infty]{}\; \mathbb{E}[y \mid x]. $$

The estimate is unbiased and its variance shrinks as $\operatorname{Var}(\hat{y}) = \sigma^2(x)/n$,
so a larger `n_samples` (in `GenerationConfig`) trades compute for a steadier prediction.

distilgpt2 is a tiny base model, so treat this as an API demonstration, not a strong regressor.
The bar to beat is the **baseline that always predicts the training mean** $\bar{y}$, whose error
we print alongside the model's.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

from sklm import DiscretizationConfig, LanguageModelRegressor, TrainingConfig

sns.set_theme(style="whitegrid", context="notebook")
SEED = 42

## Data

The diabetes dataset: ten physiological measurements per patient and a continuous disease-
progression score one year later. We keep the unscaled features (`scaled=False`) so the
serialized numbers are human-readable.

In [ ]:
data = load_diabetes(as_frame=True, scaled=False)
X = data.data.round(3)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train.head()

## Fine-tuning

`loss_on_target_only=True` restricts the training loss to the target tokens, so optimization
focuses on predicting $y$ rather than re-modelling the feature text. `discretization` snaps the
generated target onto a numeric grid before decoding, which can steady the Monte-Carlo
estimate.

In [ ]:
reg = LanguageModelRegressor(
    model="gabfssilva/distilgpt2",
    backend="mlx",
    precision="fp32",
    training=TrainingConfig(
        epochs=3, 
        augmentation_factor=4, 
        batch_size=1, 
        learning_rate=1e-5, 
        lr_scheduler="constant", 
        loss_on_target_only=True
    ),
    discretization=DiscretizationConfig(bins=1.0),
    random_state=SEED,
).fit(X_train, y_train)

## Metrics

Mean absolute error (MAE) and root mean squared error (RMSE) are in the target's units; RMSE
penalizes large misses more. The coefficient of determination compares the model to the mean
baseline,

$$ R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}, $$

so $R^2 = 1$ is perfect, $R^2 = 0$ ties the baseline, and $R^2 < 0$ means the model does *worse*
than always predicting the training mean.

In [ ]:
pred = reg.predict(X_test)
baseline = np.full(len(y_test), y_train.mean())

print(f"model MAE : {mean_absolute_error(y_test, pred):7.2f}")
print(f"model RMSE: {root_mean_squared_error(y_test, pred):7.2f}")
print(f"model R2  : {r2_score(y_test, pred):7.2f}")
print(f"mean  MAE : {mean_absolute_error(y_test, baseline):7.2f}  (baseline: predict the training mean)")

## Predicted vs. true

Points on the dashed line $\hat{y} = y$ are perfect predictions; vertical distance from it is the
error.

In [ ]:
lo = float(min(y_test.min(), pred.min()))
hi = float(max(y_test.max(), pred.max()))

fig, ax = plt.subplots(figsize=(5, 5))
sns.scatterplot(x=y_test, y=pred, ax=ax, alpha=0.7)
ax.plot([lo, hi], [lo, hi], color="crimson", linestyle="--", label="ideal (y_hat = y)")
ax.set_xlabel("true target")
ax.set_ylabel("prediction")
ax.set_title("Predicted vs. true")
ax.legend()
plt.tight_layout()
plt.show()

## Residuals

Residuals $\hat{y} - y$ should scatter around zero with no trend. A slope or fan shape signals
systematic bias the model is missing.

In [ ]:
resid = pred - y_test.to_numpy()

fig, ax = plt.subplots(figsize=(6, 4))
sns.scatterplot(x=pred, y=resid, ax=ax, alpha=0.7)
ax.axhline(0, color="crimson", linestyle="--")
ax.set_xlabel("prediction")
ax.set_ylabel("residual (y_hat - y)")
ax.set_title("Residuals")
plt.tight_layout()
plt.show()